# Outlier Detection (IQR Method)
Uses the Interquartile Range method to flag outliers in numeric columns.

**Configure the cell below**, then **Run All**. The final cell adds outlier flag columns — run it manually.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
TABLE_NAME = "{{TABLE_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
# Columns to check for outliers
NUMERIC_COLUMNS = {{NUMERIC_COLUMNS}}  # e.g., ["precio", "cantidad", "importe"]
# IQR multiplier: 1.5 = standard, 3.0 = extreme only
IQR_MULTIPLIER = {{IQR_MULTIPLIER}}
OUTPUT_SUFFIX = "_cleaned"
# ───────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import percentile_approx
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()
# Read from _cleaned if it exists (previous fixes), else original
try:
    df = spark.table(f"{TABLE_NAME}{OUTPUT_SUFFIX}")
    print(f"Reading from {TABLE_NAME}{OUTPUT_SUFFIX} (previous fixes applied)")
except:
    df = spark.table(TABLE_NAME)
    print(f"Reading from {TABLE_NAME} (original table)")
original_count = df.count()

print(f"Table: {TABLE_NAME}")
print(f"Rows: {original_count:,}")
print(f"IQR Multiplier: {IQR_MULTIPLIER}")

In [ ]:
print("=" * 60)
print(f"OUTLIER DETECTION (IQR × {IQR_MULTIPLIER})")
print("=" * 60)

outlier_results = []
outlier_bounds = {}

for col_name in NUMERIC_COLUMNS:
    numeric_df = df.where(F.col(col_name).isNotNull()).withColumn(
        "_val", F.col(col_name).cast(DoubleType())
    ).where(F.col("_val").isNotNull())

    count_val = numeric_df.count()
    if count_val < 4:
        print(f"\n'{col_name}': Too few values ({count_val}) for outlier detection — skipping.")
        continue

    quantiles = numeric_df.agg(
        percentile_approx("_val", 0.25).alias("q1"),
        percentile_approx("_val", 0.75).alias("q3")
    ).collect()[0]

    q1 = quantiles["q1"]
    q3 = quantiles["q3"]
    iqr = q3 - q1
    lower_bound = q1 - (IQR_MULTIPLIER * iqr)
    upper_bound = q3 + (IQR_MULTIPLIER * iqr)
    outlier_bounds[col_name] = (lower_bound, upper_bound)

    outliers = numeric_df.where(
        (F.col("_val") < lower_bound) | (F.col("_val") > upper_bound)
    )
    outlier_count = outliers.count()
    outlier_pct = round(outlier_count / count_val * 100, 2)

    outlier_results.append({
        "column": col_name, "q1": round(q1, 4), "q3": round(q3, 4),
        "iqr": round(iqr, 4), "lower_bound": round(lower_bound, 4),
        "upper_bound": round(upper_bound, 4),
        "outlier_count": outlier_count, "outlier_pct": outlier_pct
    })

    print(f"\n── {col_name} ──")
    print(f"  Q1={round(q1, 2)}, Q3={round(q3, 2)}, IQR={round(iqr, 2)}")
    print(f"  Bounds: [{round(lower_bound, 2)}, {round(upper_bound, 2)}]")
    print(f"  Outliers: {outlier_count:,} ({outlier_pct}%)")

    if outlier_count > 0:
        print("  Sample outliers:")
        display(outliers.select(col_name, "_val").orderBy("_val").limit(10))

if outlier_results:
    print("\n── Outlier Summary ──")
    display(spark.createDataFrame(outlier_results))

---
## ⚠ Apply Fix — Add Outlier Flags
The cell below adds boolean `{column}_outlier_flag` columns. Outlier rows are **NOT removed**, only flagged.

**Review the analysis above before running this cell.**

In [ ]:
# ⚠ APPLY FIX — Run manually after review
cleaned_df = df

for col_name, (lower, upper) in outlier_bounds.items():
    flag_col = f"{col_name}_outlier_flag"
    numeric_val = F.col(col_name).cast(DoubleType())
    cleaned_df = cleaned_df.withColumn(
        flag_col,
        F.when(
            numeric_val.isNotNull() & ((numeric_val < lower) | (numeric_val > upper)),
            F.lit(True)
        ).otherwise(F.lit(False))
    )

output_table = f"{TABLE_NAME}{OUTPUT_SUFFIX}"
cleaned_df.write.mode("overwrite").format("delta").saveAsTable(
    output_table
)

print(f"Added outlier flags for {len(outlier_bounds)} columns")
print(f"Output: {output_table}")